In [3]:
# ================================================================
# USED CAR RESALE DATASET - COMPLETE DATA PREPROCESSING WORKFLOW
# ================================================================

# -----------------------------
# 1. IMPORT REQUIRED LIBRARIES
# -----------------------------
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

from IPython.display import display

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


# ================================================================
# 2. LOAD THE DATASET
# ================================================================

# For Google Colab:
# Upload the CSV file using the file upload option, then use its name.
#
# For the provided dataset:
file_path = "Day12_Used_Car_Preprocessing_Dataset.csv"
df = pd.read_csv(file_path)


print("=" * 70)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 70)

print("Dataset Shape:", df.shape)
print("\nFirst 5 Records:")
display(df.head())


# ================================================================
# 3. INITIAL DATA INSPECTION
# ================================================================

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nDataset Information:")
df.info()

print("\nStatistical Summary:")
display(df.describe(include="all").T)


# ================================================================
# 4. CHECK MISSING VALUES
# ================================================================

print("\n" + "=" * 70)
print("MISSING VALUE ANALYSIS")
print("=" * 70)

missing_count = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Percentage": missing_percentage
})

display(missing_summary)

# Replace blank strings with NaN
df = df.replace(r"^\s*$", np.nan, regex=True)

print("\nMissing values after converting blank entries to NaN:")
display(df.isnull().sum().to_frame("Missing_Count"))


# ================================================================
# 5. CHECK DUPLICATE RECORDS
# ================================================================

print("\n" + "=" * 70)
print("DUPLICATE RECORD ANALYSIS")
print("=" * 70)

duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

if duplicate_count > 0:
    print("\nDuplicate Records:")
    display(df[df.duplicated(keep=False)])

    # Remove duplicate rows
    df = df.drop_duplicates().reset_index(drop=True)

    print("Duplicates removed.")
else:
    print("No duplicate records found.")


# ================================================================
# 6. IDENTIFY TARGET AND FEATURES
# ================================================================

target_column = "Resale_Price_Lakh"

# Car_ID is an identifier and should not be used as a predictive feature
id_columns = ["Car_ID"]

X = df.drop(columns=[target_column] + id_columns)
y = df[target_column]

print("\n" + "=" * 70)
print("FEATURE / TARGET SEPARATION")
print("=" * 70)

print("Target Variable:", target_column)
print("Identifier Removed:", id_columns)

print("\nFeature Columns:")
print(X.columns.tolist())

print("\nTarget Sample:")
display(y.head())


# ================================================================
# 7. TRAIN-TEST SPLIT
# ================================================================
# IMPORTANT:
# The split is performed BEFORE learning imputation, outlier limits,
# encoding, or scaling parameters.
#
# This prevents information from the test set leaking into training.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\n" + "=" * 70)
print("TRAIN-TEST SPLIT")
print("=" * 70)

print("Training Features Shape:", X_train.shape)
print("Testing Features Shape :", X_test.shape)
print("Training Target Shape  :", y_train.shape)
print("Testing Target Shape   :", y_test.shape)


# ================================================================
# 8. DEFINE COLUMN TYPES
# ================================================================

# Numerical variables
numeric_features = [
    "Year",
    "Mileage_Km",
    "Engine_CC",
    "Power_BHP",
    "Previous_Owners",
    "Accidents_Reported",
    "Service_Score"
]

# Ordinal variable
# Condition has a natural order:
# Poor < Fair < Good < Very Good < Excellent
ordinal_features = ["Condition"]

ordinal_categories = [
    ["Poor", "Fair", "Good", "Very Good", "Excellent"]
]

# Nominal categorical variables
nominal_features = [
    "Brand",
    "Fuel_Type",
    "Transmission",
    "City",
    "Seller_Type"
]


# ================================================================
# 9. IQR OUTLIER DETECTION AND TREATMENT
# ================================================================
# The following custom transformer calculates Q1, Q3 and IQR ONLY
# from the training data.
#
# Outliers are handled using IQR capping:
#
# Lower Limit = Q1 - 1.5 * IQR
# Upper Limit = Q3 + 1.5 * IQR
#
# Values outside these limits are capped rather than deleted.
# This preserves the number of observations.

class IQRClipper(BaseEstimator, TransformerMixin):

    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):
        X = pd.DataFrame(X)

        self.lower_bounds_ = X.quantile(0.25) - (
            self.factor * (X.quantile(0.75) - X.quantile(0.25))
        )

        self.upper_bounds_ = X.quantile(0.75) + (
            self.factor * (X.quantile(0.75) - X.quantile(0.25))
        )

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()

        X = X.clip(
            lower=self.lower_bounds_,
            upper=self.upper_bounds_,
            axis=1
        )

        return X.values


# ================================================================
# 10. NUMERICAL PREPROCESSING PIPELINE
# ================================================================
# Steps:
# 1. Fill missing numerical values with median.
# 2. Detect and cap outliers using IQR.
# 3. Standardize numerical features.
#
# All parameters are learned only from X_train.

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "outlier_treatment",
            IQRClipper(factor=1.5)
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ================================================================
# 11. ORDINAL PREPROCESSING PIPELINE
# ================================================================
# Condition is ordinal because its categories have a meaningful order.

ordinal_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "ordinal_encoder",
            OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )
        )
    ]
)


# ================================================================
# 12. NOMINAL PREPROCESSING PIPELINE
# ================================================================
# Nominal variables do not have a natural order.
#
# One-Hot Encoding is therefore used.
#
# handle_unknown="ignore" prevents errors if a category appears in
# the test set that was not present in the training set.

nominal_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot_encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)


# ================================================================
# 13. COMBINE ALL PREPROCESSING STEPS
# ================================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "ordinal",
            ordinal_pipeline,
            ordinal_features
        ),
        (
            "nominal",
            nominal_pipeline,
            nominal_features
        )
    ],
    remainder="drop"
)


# ================================================================
# 14. FIT PREPROCESSING ONLY ON TRAINING DATA
# ================================================================

print("\n" + "=" * 70)
print("FITTING PREPROCESSING PIPELINE")
print("=" * 70)

# IMPORTANT:
# fit_transform() is used ONLY for training data.
# Therefore, medians, IQR limits, category mappings and scaling
# parameters are learned only from X_train.

X_train_processed = preprocessor.fit_transform(X_train)

# Test data is ONLY transformed using parameters learned from X_train.
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed successfully.")

print("\nProcessed Training Shape:", X_train_processed.shape)
print("Processed Testing Shape :", X_test_processed.shape)


# ================================================================
# 15. GET PROCESSED FEATURE NAMES
# ================================================================
# ================================================================
# 15. GET PROCESSED FEATURE NAMES
# ================================================================

try:
    feature_names = preprocessor.get_feature_names_out()

    # Clean feature names for easier readability
    feature_names = [
        name.replace("num__", "")
            .replace("cat__", "")
            .replace("onehot__", "")
            .replace("remainder__", "")
        for name in feature_names
    ]

    print("Processed Feature Names:")
    for i, name in enumerate(feature_names, 1):
        print(f"{i}. {name}")

except Exception as e:
    print("Could not automatically retrieve feature names.")
    print("Error:", e)

    # Fallback: create generic feature names
    feature_names = [
        f"Feature_{i+1}"
        for i in range(X_train_processed.shape[1])
    ]

    print("\nUsing generic feature names:")
    print(feature_names)
    
# ================================================================
# 16. CONVERT PROCESSED DATA TO DATAFRAMES
# ================================================================

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

# Reset indexes for clean output files
X_train_processed_df = X_train_processed_df.reset_index(drop=True)
X_test_processed_df = X_test_processed_df.reset_index(drop=True)

y_train_processed = y_train.reset_index(drop=True)
y_test_processed = y_test.reset_index(drop=True)


# ================================================================
# 17. CREATE FINAL PREPROCESSED TRAINING AND TESTING DATASETS
# ================================================================

train_processed = X_train_processed_df.copy()
train_processed[target_column] = y_train_processed

test_processed = X_test_processed_df.copy()
test_processed[target_column] = y_test_processed


# ================================================================
# 18. DISPLAY PROCESSED DATA
# ================================================================

print("\n" + "=" * 70)
print("PROCESSED TRAINING DATA")
print("=" * 70)

display(train_processed.head())

print("\nProcessed Training Dataset Shape:")
print(train_processed.shape)


print("\n" + "=" * 70)
print("PROCESSED TESTING DATA")
print("=" * 70)

display(test_processed.head())

print("\nProcessed Testing Dataset Shape:")
print(test_processed.shape)


# ================================================================
# 19. VERIFY MISSING VALUES AFTER PREPROCESSING
# ================================================================

print("\n" + "=" * 70)
print("MISSING VALUE VERIFICATION")
print("=" * 70)

print("Missing values in processed training data:")
print(train_processed.isnull().sum().sum())

print("Missing values in processed testing data:")
print(test_processed.isnull().sum().sum())


# ================================================================
# 20. VERIFY DUPLICATES
# ================================================================

print("\n" + "=" * 70)
print("DUPLICATE VERIFICATION")
print("=" * 70)

print(
    "Duplicate rows in processed training data:",
    train_processed.duplicated().sum()
)

print(
    "Duplicate rows in processed testing data:",
    test_processed.duplicated().sum()
)


# ================================================================
# 21. VERIFY SCALING
# ================================================================
# Numerical features should have approximately:
# Mean = 0
# Standard Deviation = 1
#
# Small deviations can occur depending on the data and transformations.

processed_numeric_columns = [
    col for col in feature_names
    if col in numeric_features
]

print("\n" + "=" * 70)
print("SCALING VERIFICATION")
print("=" * 70)

print("Training numerical feature means:")
display(
    train_processed[processed_numeric_columns].mean().round(4)
)

print("\nTraining numerical feature standard deviations:")
display(
    train_processed[processed_numeric_columns].std().round(4)
)


# ================================================================
# 22. VERIFY TARGET VARIABLE
# ================================================================
# The target variable is intentionally NOT scaled.
# It remains in its original Resale_Price_Lakh units.

print("\n" + "=" * 70)
print("TARGET VARIABLE VERIFICATION")
print("=" * 70)

print("Target variable:", target_column)

print("\nTraining target statistics:")
display(y_train_processed.describe())

print("\nTesting target statistics:")
display(y_test_processed.describe())


# ================================================================
# 23. OUTLIER SUMMARY FROM TRAINING DATA
# ================================================================
# Report how many observations were outside the IQR limits BEFORE
# capping. The limits were calculated exclusively from X_train.

print("\n" + "=" * 70)
print("IQR OUTLIER SUMMARY")
print("=" * 70)

# Median imputation first so the IQR calculation is valid
train_numeric_for_outliers = X_train[numeric_features].copy()

for col in numeric_features:
    train_numeric_for_outliers[col] = train_numeric_for_outliers[col].fillna(
        train_numeric_for_outliers[col].median()
    )

Q1 = train_numeric_for_outliers.quantile(0.25)
Q3 = train_numeric_for_outliers.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_counts = {}

for col in numeric_features:
    outlier_counts[col] = (
        (
            (train_numeric_for_outliers[col] < lower_bound[col]) |
            (train_numeric_for_outliers[col] > upper_bound[col])
        )
        .sum()
    )

outlier_summary = pd.DataFrame({
    "Q1": Q1,
    "Q3": Q3,
    "IQR": IQR,
    "Lower_Bound": lower_bound,
    "Upper_Bound": upper_bound,
    "Outlier_Count": pd.Series(outlier_counts)
})

display(outlier_summary)


# ================================================================
# 24. SAVE PREPROCESSED DATASETS
# ================================================================

train_output = "Used_Car_Preprocessed_Train.csv"
test_output = "Used_Car_Preprocessed_Test.csv"
full_output = "Used_Car_Preprocessed_Dataset.csv"

train_processed.to_csv(train_output, index=False)
test_processed.to_csv(test_output, index=False)

# Combine the processed training and testing datasets
full_processed = pd.concat(
    [train_processed, test_processed],
    ignore_index=True
)

full_processed.to_csv(full_output, index=False)


# ================================================================
# 25. SAVE THE PREPROCESSING PIPELINE
# ================================================================
# This allows the exact preprocessing process to be reused later.

import joblib

pipeline_output = "Used_Car_Preprocessing_Pipeline.pkl"

joblib.dump(preprocessor, pipeline_output)


# ================================================================
# 26. FINAL VERIFICATION
# ================================================================

print("\n" + "=" * 70)
print("FINAL PREPROCESSING VERIFICATION")
print("=" * 70)

print("\nOriginal dataset shape:", df.shape)
print("Processed training shape:", train_processed.shape)
print("Processed testing shape :", test_processed.shape)
print("Processed full dataset shape:", full_processed.shape)

print("\nTotal missing values in final dataset:",
      full_processed.isnull().sum().sum())

print("\nTotal duplicate rows in final dataset:",
      full_processed.duplicated().sum())

print("\nFinal processed columns:")
print(full_processed.columns.tolist())


# ================================================================
# 27. DISPLAY FINAL DATASET
# ================================================================

print("\n" + "=" * 70)
print("FINAL PREPROCESSED DATASET - FIRST 10 ROWS")
print("=" * 70)

display(full_processed.head(10))


# ================================================================
# 28. DISPLAY OUTPUT FILE LOCATIONS
# ================================================================

print("\n" + "=" * 70)
print("OUTPUT FILES CREATED")
print("=" * 70)

print("1. Training dataset :", train_output)
print("2. Testing dataset  :", test_output)
print("3. Full preprocessed dataset :", full_output)
print("4. Preprocessing pipeline :", pipeline_output)

print("\nPreprocessing workflow completed successfully!")


# ================================================================
# PREPROCESSING DECISIONS SUMMARY
# ================================================================
#
# 1. Car_ID:
#    Removed because it is only an identifier and does not provide
#    useful predictive information.
#
# 2. Missing Values:
#    Numerical columns -> Median imputation.
#    Categorical columns -> Most-frequent imputation.
#
# 3. Duplicate Records:
#    Duplicate rows are identified and removed before splitting.
#
# 4. Outliers:
#    IQR method is used on numerical predictor variables.
#    Outlier values are capped at:
#
#       Lower = Q1 - 1.5 * IQR
#       Upper = Q3 + 1.5 * IQR
#
#    The IQR limits are learned ONLY from the training data.
#
# 5. Condition:
#    Ordinal encoding is used because:
#
#       Poor < Fair < Good < Very Good < Excellent
#
# 6. Nominal Categorical Variables:
#    Brand, Fuel_Type, Transmission, City and Seller_Type
#    are converted using One-Hot Encoding.
#
# 7. Numerical Variables:
#    StandardScaler is used so numerical predictors have approximately
#    zero mean and unit variance.
#
# 8. Target:
#    Resale_Price_Lakh is kept separate and is NOT scaled.
#
# 9. Train/Test Split:
#    80% training and 20% testing data with random_state=42.
#
# 10. Data Leakage Prevention:
#     All preprocessing operations that learn parameters are fitted
#     only on the training dataset:
#
#       - Median imputation
#       - IQR outlier limits
#       - Ordinal encoding
#       - One-hot encoding
#       - StandardScaler
#
#     The test set is transformed only after the preprocessing
#     pipeline has been fitted on the training data.
#
# ================================================================

DATASET LOADED SUCCESSFULLY
Dataset Shape: (320, 15)

First 5 Records:


,Car_ID,Brand,Year,Mileage_Km,Engine_CC,Power_BHP,Fuel_Type,Transmission,City,Seller_Type,Condition,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
0,CAR0001,Skoda,2021,69708,1152,128.8,Diesel,Manual,Lucknow,Individual,Good,1,0,72,6.38
1,CAR0002,Toyota,2020,88881,903,146.5,Diesel,Automatic,Chandigarh,Individual,Good,1,0,87,4.83
2,CAR0003,Volkswagen,2021,43646,1446,185.9,Diesel,Automatic,Hyderabad,Individual,Very Good,2,0,90,7.30
3,CAR0004,Tata,2019,70847,2069,148.8,Petrol,Manual,Lucknow,Individual,Excellent,3,0,66,3.82
4,CAR0005,Tata,2016,101228,1657,206.0,Petrol,Automatic,Ahmedabad,Dealer,Very Good,2,0,84,1.93



DATASET INFORMATION

Column Names:
['Car_ID', 'Brand', 'Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type', 'Condition', 'Previous_Owners', 'Accidents_Reported', 'Service_Score', 'Resale_Price_Lakh']

Data Types:
Car_ID                 object
Brand                  object
Year                    int64
Mileage_Km              int64
Engine_CC               int64
Power_BHP             float64
Fuel_Type              object
Transmission           object
City                   object
Seller_Type            object
Condition              object
Previous_Owners         int64
Accidents_Reported      int64
Service_Score           int64
Resale_Price_Lakh     float64
dtype: object

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Car_ID              320 non-null    object 
 

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Car_ID,320,320,CAR0001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Brand,320,10,Volkswagen,39,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Year,320.0,NaN,NaN,NaN,2019.5375,3.341367,2014.0,2017.0,2020.0,2022.0,2025.0
Mileage_Km,320.0,NaN,NaN,NaN,74110.203125,38885.260771,700.0,46323.25,72718.5,97951.5,320000.0
Engine_CC,320.0,NaN,NaN,NaN,1346.703125,543.40816,600.0,1004.75,1303.0,1635.25,5000.0
Power_BHP,320.0,NaN,NaN,NaN,150.489688,36.665353,51.4,128.45,150.75,171.475,390.0
Fuel_Type,320,4,Petrol,182,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Transmission,320,2,Manual,197,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,320,10,Lucknow,43,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Seller_Type,320,3,Individual,167,NaN,NaN,NaN,NaN,NaN,NaN,NaN



MISSING VALUE ANALYSIS


...


IQR OUTLIER SUMMARY


,Q1,Q3,IQR,Lower_Bound,Upper_Bound,Outlier_Count
Year,2017.000,2022.00,5.000,2009.5000,2029.5000,0
Mileage_Km,46076.750,97695.75,51619.000,-31351.7500,175124.2500,2
Engine_CC,1013.250,1644.75,631.500,66.0000,2592.0000,6
Power_BHP,129.675,171.15,41.475,67.4625,233.3625,5
Previous_Owners,1.000,2.00,1.000,-0.5000,3.5000,10
Accidents_Reported,0.000,0.00,0.000,0.0000,0.0000,47
Service_Score,65.750,86.00,20.250,35.3750,116.3750,0



FINAL PREPROCESSING VERIFICATION

Original dataset shape: (320, 15)
Processed training shape: (256, 38)
Processed testing shape : (64, 38)
Processed full dataset shape: (320, 38)

Total missing values in final dataset: 0

Total duplicate rows in final dataset: 0

Final processed columns:
['Feature_1', 'Feature_2', 'Feature_3', 'Feature_4', 'Feature_5', 'Feature_6', 'Feature_7', 'Feature_8', 'Feature_9', 'Feature_10', 'Feature_11', 'Feature_12', 'Feature_13', 'Feature_14', 'Feature_15', 'Feature_16', 'Feature_17', 'Feature_18', 'Feature_19', 'Feature_20', 'Feature_21', 'Feature_22', 'Feature_23', 'Feature_24', 'Feature_25', 'Feature_26', 'Feature_27', 'Feature_28', 'Feature_29', 'Feature_30', 'Feature_31', 'Feature_32', 'Feature_33', 'Feature_34', 'Feature_35', 'Feature_36', 'Feature_37', 'Resale_Price_Lakh']

FINAL PREPROCESSED DATASET - FIRST 10 ROWS


,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Feature_7,Feature_8,Feature_9,Feature_10,Feature_11,Feature_12,Feature_13,Feature_14,Feature_15,Feature_16,Feature_17,Feature_18,Feature_19,Feature_20,Feature_21,Feature_22,Feature_23,Feature_24,Feature_25,Feature_26,Feature_27,Feature_28,Feature_29,Feature_30,Feature_31,Feature_32,Feature_33,Feature_34,Feature_35,Feature_36,Feature_37,Resale_Price_Lakh
0,-0.486391,-0.225883,-0.322882,0.304485,-0.755752,0.0,-0.454105,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,4.26
1,0.725445,0.089371,-0.656431,-0.330444,-0.755752,0.0,-0.614672,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,5.30
2,-1.395268,1.115798,-0.120529,0.420784,-0.755752,0.0,-0.534389,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.23
3,1.028404,-0.195076,0.479859,0.414498,0.484456,0.0,0.188165,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,7.09
4,-1.092309,0.706680,0.786724,-0.437314,0.484456,0.0,0.107881,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.69
5,1.331363,-0.579378,-0.176121,-1.207401,0.484456,0.0,1.552989,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,5.71
6,0.725445,0.122976,-1.432489,-1.895765,-0.755752,0.0,0.509300,3.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,5.02
7,0.725445,-0.250586,0.642186,-0.795640,-0.755752,0.0,-0.855524,4.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,6.00
8,0.422486,-0.753059,0.121850,-0.229861,-0.755752,0.0,0.750151,2.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,5.27
9,0.725445,-1.287669,-1.496975,-0.773637,0.484456,0.0,1.392421,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,5.77



OUTPUT FILES CREATED
1. Training dataset : Used_Car_Preprocessed_Train.csv
2. Testing dataset  : Used_Car_Preprocessed_Test.csv
3. Full preprocessed dataset : Used_Car_Preprocessed_Dataset.csv
4. Preprocessing pipeline : Used_Car_Preprocessing_Pipeline.pkl

Preprocessing workflow completed successfully!
